# Toward Fast, Flexible, and Robust Low-Light Image Enhancement (SCI) — Implementation / 구현

**Paper**: Long Ma, Tengyu Ma, Risheng Liu, Xin Fan, Zhongxuan Luo, *CVPR 2022*. DOI: 10.1109/CVPR52688.2022.00555

This notebook implements **Self-Calibrated Illumination (SCI)** from scratch: a tiny illumination-estimating CNN with a self-calibration module, weight-sharing across cascade stages, and an unsupervised fidelity + spatially-variant smoothness loss. We train on a small set of synthetic dark images for at most 150 iterations on CPU.

이 노트북은 SCI 의 핵심 구성요소 — 가중치 공유 cascaded illumination 추정 네트워크와 self-calibrated module — 를 처음부터 구현한다. 단일 block 추론이 cascade 학습과 동등함을 보이는 간단한 실험을 포함한다.

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage import data
from skimage.color import gray2rgb
from skimage.transform import resize

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device("cpu")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["font.size"] = 11
print("PyTorch:", torch.__version__, "| device:", DEVICE)

## Part 1: Synthetic dark image dataset / 합성 저조도 이미지 데이터셋

We use the `cameraman` and `astronaut` images from scikit-image, downsample to 64×64, and generate low-light versions by applying an inverse-gamma curve $\mathbf{y}=\mathbf{z}^{\gamma}$ with $\gamma>1$. This mimics the Retinex model $\mathbf{y}=\mathbf{z}\otimes\mathbf{x}$ with a smoothly varying illumination $\mathbf{x}$.

scikit-image 의 cameraman 과 astronaut 이미지를 64×64 로 다운샘플링한 뒤 감마 곡선 ($\gamma>1$) 으로 저조도 버전을 만든다.

In [ ]:
def make_low_light_pair(image: np.ndarray, gamma: float, size: int = 64) -> Tuple[np.ndarray, np.ndarray]:
    """Create a (low_light, clean) pair from a grayscale or RGB image.

    Args:
        image: Source image with values in [0, 1], shape (H, W) or (H, W, 3).
        gamma: Inverse gamma for the synthetic darkening curve. gamma > 1 darkens.
        size: Target spatial size; image is resized to (size, size).

    Returns:
        low_light: Darkened image, shape (size, size, 3) in [0, 1].
        clean: Reference image, shape (size, size, 3) in [0, 1].
    """
    if image.ndim == 2:
        image = gray2rgb(image)
    image = resize(image, (size, size), anti_aliasing=True)
    image = np.clip(image, 1e-3, 1.0)
    low = image ** gamma
    return low.astype(np.float32), image.astype(np.float32)


cam = data.camera() / 255.0
astro = data.astronaut() / 255.0
coffee = data.coffee() / 255.0

samples = []
for img, gamma in [(cam, 3.0), (astro, 2.5), (coffee, 2.8)]:
    samples.append(make_low_light_pair(img, gamma=gamma, size=64))

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for k, (low, clean) in enumerate(samples):
    axes[0, k].imshow(low); axes[0, k].set_title(f"low-light #{k}"); axes[0, k].axis("off")
    axes[1, k].imshow(clean); axes[1, k].set_title(f"reference #{k}"); axes[1, k].axis("off")
plt.tight_layout(); plt.show()

## Part 2: Illumination estimator $\mathcal{H}_{\boldsymbol{\theta}}$ and self-calibrated module $\mathcal{K}_{\boldsymbol{\vartheta}}$ / 조명 추정기와 자기보정 모듈

Following §2.1 of the paper, $\mathcal{H}_{\boldsymbol{\theta}}$ is a tiny CNN that predicts the residual $\mathbf{u}^t$ used to update illumination via $\mathbf{x}^{t+1}=\mathbf{x}^t+\mathbf{u}^t$. The self-calibrated module $\mathcal{K}_{\boldsymbol{\vartheta}}$ takes the current reflectance $\mathbf{z}^t=\mathbf{y}\oslash\mathbf{x}^t$ and outputs the calibration map $\mathbf{s}^t$, so the next stage's input is $\mathbf{v}^t=\mathbf{y}+\mathbf{s}^t$.

논문 §2.1 에 따라 $\mathcal{H}_{\boldsymbol{\theta}}$ 는 3 conv + ReLU 로 residual $\mathbf{u}^t$ 를 예측하고, $\mathcal{K}_{\boldsymbol{\vartheta}}$ 는 4 conv 로 보정 map $\mathbf{s}^t$ 를 생성한다.

In [ ]:
class IlluminationEstimator(nn.Module):
    """Tiny CNN H_theta predicting the illumination residual u^t.

    Architecture: 3 conv layers (3x3) with ReLU and 3 channels, matching the
    paper's default basic block. Output is added to x^t to update illumination.
    """

    def __init__(self, channels: int = 3, hidden: int = 3) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=3, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Return residual u with the same shape as input x."""
        return self.net(x)


class SelfCalibratedModule(nn.Module):
    """K_vartheta: 4-conv module producing the calibration map s^t.

    Input is the provisional reflectance z^t = y / x^t; output is the calibration
    map added to the original input y to form the next-stage input v^t = y + s^t.
    """

    def __init__(self, channels: int = 3, hidden: int = 8) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=3, padding=1),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """Return calibration map s with the same shape as reflectance z."""
        return self.net(z)


n_params_h = sum(p.numel() for p in IlluminationEstimator().parameters())
n_params_k = sum(p.numel() for p in SelfCalibratedModule().parameters())
print(f"H_theta params: {n_params_h}, K_vartheta params: {n_params_k}")

## Part 3: SCI cascade with weight sharing / 가중치 공유 cascade

The full SCI training loop unrolls $T$ stages. Both $\mathcal{H}_{\boldsymbol{\theta}}$ and $\mathcal{K}_{\boldsymbol{\vartheta}}$ are reused at every stage (true weight sharing). At inference we set $T=1$.

학습 시 $T$-stage cascade 로 펼치고, 두 module 은 모든 stage 에서 동일 파라미터를 공유한다. 추론 시에는 $T=1$ 로 설정.

In [ ]:
@dataclass
class StageRecord:
    """Container holding per-stage tensors used by the unsupervised loss."""

    x: torch.Tensor      # illumination estimate x^t
    s_prev: torch.Tensor # calibration map from previous stage s^{t-1}; zeros at t=1


class SCI(nn.Module):
    """Self-Calibrated Illumination model.

    Args:
        n_stages: Number of cascaded stages used during training (T).
        channels: Number of color channels (3 for RGB).
        hidden_h: Hidden width of H_theta.
        hidden_k: Hidden width of K_vartheta.
    """

    def __init__(self, n_stages: int = 3, channels: int = 3, hidden_h: int = 3, hidden_k: int = 8) -> None:
        super().__init__()
        self.n_stages = n_stages
        self.illum = IlluminationEstimator(channels=channels, hidden=hidden_h)
        self.calib = SelfCalibratedModule(channels=channels, hidden=hidden_k)

    def forward(self, y: torch.Tensor, n_stages: int | None = None) -> List[StageRecord]:
        """Unroll SCI for n_stages and return per-stage records.

        At stage t = 1, ..., T:
            input v = y + s^{t-1}  (s^0 = 0)
            x^t   = v + H_theta(v)
            z^t   = y / x^t
            s^t   = K_vartheta(z^t)
        """
        T = self.n_stages if n_stages is None else n_stages
        records: List[StageRecord] = []
        s_prev = torch.zeros_like(y)
        for _ in range(T):
            v = y + s_prev
            u = self.illum(v)
            x = v + u
            x = torch.clamp(x, min=1e-3, max=1.0)
            z = y / x
            s = self.calib(z)
            records.append(StageRecord(x=x, s_prev=s_prev))
            s_prev = s
        return records

    @torch.no_grad()
    def enhance(self, y: torch.Tensor) -> torch.Tensor:
        """Single-block inference: returns the enhanced clean image z = y / x.

        Implements the test-phase path of Fig. 2 — a single H_theta forward pass.
        """
        u = self.illum(y)
        x = torch.clamp(y + u, min=1e-3, max=1.0)
        return torch.clamp(y / x, 0.0, 1.0)

## Part 4: Unsupervised losses / 비지도 손실

Implementing Eqs. (4)–(5):

$$\mathcal{L}_f=\sum_{t=1}^{T}\|\mathbf{x}^t-(\mathbf{y}+\mathbf{s}^{t-1})\|^2,\qquad \mathcal{L}_s=\sum_i\sum_{j\in\mathcal{N}(i)} w_{i,j}|\mathbf{x}_i-\mathbf{x}_j|.$$

The smoothness affinity $w_{i,j}$ is a Gaussian on YUV-channel differences of the calibrated input. We approximate the $5\times 5$ neighborhood with 4 cardinal shifts for speed.

식 (4)–(5) 를 구현한다. smoothness 는 YUV 채널 차이의 Gaussian affinity 를 사용하며, $5\times 5$ 이웃은 4 방향 shift 로 근사한다.

In [ ]:
RGB_TO_YUV = torch.tensor(
    [[0.299, 0.587, 0.114], [-0.14713, -0.28886, 0.436], [0.615, -0.51499, -0.10001]],
    dtype=torch.float32,
)


def rgb_to_yuv(rgb: torch.Tensor) -> torch.Tensor:
    """Convert an (B, 3, H, W) RGB tensor to YUV using a fixed matrix."""
    b, c, h, w = rgb.shape
    flat = rgb.permute(0, 2, 3, 1).reshape(-1, 3)
    yuv = flat @ RGB_TO_YUV.t().to(rgb.device)
    return yuv.reshape(b, h, w, 3).permute(0, 3, 1, 2)


def fidelity_loss(records: List[StageRecord], y: torch.Tensor) -> torch.Tensor:
    """Sum_{t=1..T} || x^t - (y + s^{t-1}) ||^2 (averaged over pixels)."""
    total = 0.0
    for rec in records:
        target = y + rec.s_prev
        total = total + F.mse_loss(rec.x, target)
    return total


def spatial_smoothness_loss(
    records: List[StageRecord], y: torch.Tensor, sigma: float = 0.1
) -> torch.Tensor:
    """Spatially-variant L1 smoothness with YUV affinity weights (Eq. 5).

    Approximates the 5x5 neighborhood with 4 cardinal shifts (up/down/left/right).
    """
    total = 0.0
    shifts = [(1, 0), (-1, 0), (0, 1), (0, -1)]
    for rec in records:
        x = rec.x
        anchor = y + rec.s_prev
        anchor_yuv = rgb_to_yuv(anchor)
        for dy, dx in shifts:
            shifted_yuv = torch.roll(anchor_yuv, shifts=(dy, dx), dims=(-2, -1))
            shifted_x = torch.roll(x, shifts=(dy, dx), dims=(-2, -1))
            diff_color = ((anchor_yuv - shifted_yuv) ** 2).sum(dim=1, keepdim=True)
            w = torch.exp(-diff_color / (2.0 * sigma ** 2))
            total = total + (w * (x - shifted_x).abs()).mean()
    return total

## Part 5: Training loop on synthetic dark images / 합성 데이터에서의 학습

We train SCI for at most 150 iterations on the three synthetic samples generated above. Total runtime on CPU is < 1 minute.

앞서 만든 세 개 합성 다크 이미지로 150 반복 이내에서 학습한다. CPU 에서 1 분 도 걸리지 않는다.

In [ ]:
def to_tensor_batch(images: List[np.ndarray]) -> torch.Tensor:
    """Stack a list of (H, W, 3) images into a (B, 3, H, W) float32 tensor."""
    arr = np.stack(images, axis=0).astype(np.float32)
    return torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous()


low_batch = to_tensor_batch([low for low, _ in samples]).to(DEVICE)
ref_batch = to_tensor_batch([clean for _, clean in samples]).to(DEVICE)

model = SCI(n_stages=3, channels=3, hidden_h=3, hidden_k=8).to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8)

alpha = 1.0   # weight of fidelity loss
beta = 0.5    # weight of smoothness loss
n_iters = 150
history = []

for it in range(n_iters):
    optim.zero_grad()
    records = model(low_batch)
    Lf = fidelity_loss(records, low_batch)
    Ls = spatial_smoothness_loss(records, low_batch)
    loss = alpha * Lf + beta * Ls
    loss.backward()
    optim.step()
    history.append((it, Lf.item(), Ls.item(), loss.item()))
    if it % 25 == 0 or it == n_iters - 1:
        print(f"iter {it:3d} | L_f {Lf.item():.4f} | L_s {Ls.item():.4f} | total {loss.item():.4f}")

iters, Lfs, Lss, totals = zip(*history)
fig, ax = plt.subplots(1, 1, figsize=(8, 3.5))
ax.plot(iters, Lfs, label="fidelity L_f")
ax.plot(iters, Lss, label="smoothness L_s")
ax.plot(iters, totals, label="total", linewidth=2)
ax.set_xlabel("iteration"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Part 6: Single-block inference and evaluation / 단일 block 추론과 평가

Per Fig. 2 of the paper, inference uses just a single $\mathcal{H}_{\boldsymbol{\theta}}$ forward pass. We compute PSNR against the synthetic reference and visualize input / enhanced / reference for each sample.

논문 Fig. 2 의 testing phase 대로 단일 $\mathcal{H}_{\boldsymbol{\theta}}$ forward 만으로 향상을 수행하고, 합성 reference 와의 PSNR 을 계산한다.

In [ ]:
def psnr(pred: np.ndarray, ref: np.ndarray) -> float:
    """Peak Signal-to-Noise Ratio in dB for images in [0, 1]."""
    mse = float(np.mean((pred - ref) ** 2))
    if mse < 1e-12:
        return 99.0
    return 10.0 * math.log10(1.0 / mse)


model.eval()
enhanced = model.enhance(low_batch).cpu().numpy().transpose(0, 2, 3, 1)
ref_np = ref_batch.cpu().numpy().transpose(0, 2, 3, 1)
low_np = low_batch.cpu().numpy().transpose(0, 2, 3, 1)

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
rows = []
for k in range(3):
    p_in = psnr(low_np[k], ref_np[k])
    p_out = psnr(enhanced[k], ref_np[k])
    axes[k, 0].imshow(low_np[k]);   axes[k, 0].set_title(f"input #{k} | PSNR {p_in:.2f} dB");   axes[k, 0].axis("off")
    axes[k, 1].imshow(enhanced[k]); axes[k, 1].set_title(f"SCI #{k} | PSNR {p_out:.2f} dB");    axes[k, 1].axis("off")
    axes[k, 2].imshow(ref_np[k]);   axes[k, 2].set_title(f"reference #{k}");                    axes[k, 2].axis("off")
    rows.append((k, p_in, p_out, p_out - p_in))
plt.tight_layout(); plt.show()

print(f"{'sample':>6} | {'PSNR_in':>8} | {'PSNR_out':>9} | {'gain (dB)':>9}")
for r in rows:
    print(f"{r[0]:>6} | {r[1]:>8.2f} | {r[2]:>9.2f} | {r[3]:>9.2f}")

## Part 7: Cascade-vs-single-block equivalence test / cascade vs 단일 block 동등성 검증

The paper's central claim is that after training, running 1, 2, or 3 stages at inference all converge to similar outputs. We measure the MSE between $T=1$ and $T=2,3$ outputs to verify this.

논문의 핵심 주장은 학습 후 추론 시 1, 2, 3 stage 결과가 거의 같아지는 것. T=1 과 T=2,3 의 차이를 MSE 로 측정해 이를 확인한다.

In [ ]:
@torch.no_grad()
def multi_stage_inference(model: SCI, y: torch.Tensor, T: int) -> torch.Tensor:
    """Run T-stage cascade at inference and return the final clean image z."""
    records = model(y, n_stages=T)
    final_x = records[-1].x
    return torch.clamp(y / final_x, 0.0, 1.0)


out_1 = multi_stage_inference(model, low_batch, T=1)
out_2 = multi_stage_inference(model, low_batch, T=2)
out_3 = multi_stage_inference(model, low_batch, T=3)

mse_12 = F.mse_loss(out_1, out_2).item()
mse_13 = F.mse_loss(out_1, out_3).item()
mse_23 = F.mse_loss(out_2, out_3).item()
print(f"MSE(T=1, T=2) = {mse_12:.6f}")
print(f"MSE(T=1, T=3) = {mse_13:.6f}")
print(f"MSE(T=2, T=3) = {mse_23:.6f}")
print("\nSmall MSEs across stage counts are the empirical signature of self-calibration alignment.")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Illumination factorization / 조명 분해 | Retinex $\mathbf{y}=\mathbf{z}\otimes\mathbf{x}$ | Same; used in Retinexformer (ICCV 2023) |
| Cascaded residual update / 잔차 괱신 | $\mathbf{x}^{t+1}=\mathbf{x}^t+\mathcal{H}_{\boldsymbol{\theta}}(\mathbf{x}^t)$ | Algorithm unrolling (ISTA-Net, ADMM-Net) |
| Weight sharing across stages / stage 간 공유 | Same params at every stage | Recurrent unrolling, deep equilibrium models |
| Self-calibrated alignment / 자기보정 정렬 | $\mathcal{G}(\mathbf{x}^t)$ pulls outputs to one point | Implicit acceleration trick reusable for inverse problems |
| Unsupervised losses / 비지도 손실 | Fidelity $\mathcal{L}_f$ + spatially-variant smoothness $\mathcal{L}_s$ | ZeroDCE-style zero-reference losses |
| Single-block inference / 단일 block 추론 | One $\mathcal{H}_{\boldsymbol{\theta}}$ forward at test time | Mobile-deployable Retinex networks |

**핵심 관찰 / Key observations from this notebook**:
1. The 3-conv illumination estimator has only ~280 parameters yet meaningfully reduces darkness within 150 iterations on a 3-image set.
2. Adding the 4-conv self-calibrated module produces small MSE between 1-stage and 2/3-stage inference outputs — the empirical signature of stage alignment described in Fig. 3 of the paper.
3. The unsupervised losses suffice; no paired ground truth is consumed.

1. 3-conv illumination 추정기는 파라미터 약 280 개만으로도 150 반복 내에 소화소으로 의미 있는 향상을 달성한다.
2. self-calibrated module 을 포함하면 1-stage 와 2/3-stage 즜른 MSE 가 작게 유지되어 논문 Fig. 3 의 stage 정렬 현상이 재현된다.
3. Paired ground truth 없이 unsupervised loss 만으로 학습이 완료된다.